In [1]:
import numpy as np
import pandas as pd

from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
x, y = make_regression(
    n_samples=500,
    n_features=10,
    n_informative=5,
    noise=20,
    random_state=42
)

In [3]:
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42
)

In [4]:
lasso = Lasso(alpha=0.1)

lasso.fit(x_train, y_train)

,"alpha alpha: float, default=1.0Constant that multiplies the L1 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Lasso` object is not advised.Instead, you should use the :class:`LinearRegression` object.",0.1
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"precompute precompute: bool or array-like of shape (n_features, n_features), default=FalseWhether to use a precomputed Gram matrix to speed upcalculations. The Gram matrix can also be passed as argument.For sparse input this option is always ``False`` to preserve sparsity.",False
,"copy_X copy_X: bool, default=TrueIf ``True``, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=1000The maximum number of iterations.",1000
,"tol tol: float, default=1e-4The tolerance for the optimization: if the updates are smaller or equal to``tol``, the optimization code checks the dual gap for optimality and continuesuntil it is smaller or equal to ``tol``, see Notes below.",0.0001
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fit asinitialization, otherwise, just erase the previous solution.See :term:`the Glossary <warm_start>`.",False
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.",False
,"random_state random_state: int, RandomState instance, default=NoneThe seed of the pseudo random number generator that selects a randomfeature to update. Used when ``selection`` == 'random'.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",None
,"selection selection: {'cyclic', 'random'}, default='cyclic'If set to 'random', a random coefficient is updated every iterationrather than looping over features sequentially by default. This(setting to 'random') often leads to significantly faster convergenceespecially when tol is higher than 1e-4.",'cyclic'
Name,Type,Value


In [9]:
y_train_pred = lasso.predict(x_train)
y_pred = lasso.predict(x_test)

In [10]:
print("MSE:", mean_squared_error(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))

MSE: 466.5397682901855
R2: 0.9017096646572718


In [11]:
print(lasso.coef_)

[28.89393921 -0.05796303 19.5781544   0.95237845  0.58691157 -0.33532908
 45.39162925 24.89600725 15.73582422 -0.93577407]


In [23]:
alphas = [0.01, 0.1, 1, 10, 100]

for alpha in alphas:
    model = Lasso(alpha=alpha)
    model.fit(x_train, y_train)

    y_train_pred = model.predict(x_train)
    y_pred = model.predict(x_test)

    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_pred)

    zero_coefficients = np.sum(model.coef_ == 0)

    print(
        f"Alpha={alpha:<4} | "
        f"Zero coefficients={zero_coefficients:<4} | "
        f"Train R2={train_r2:.4f} | "
        f"Test R2={test_r2:.4f}"
    )

Alpha=0.01 | Zero coefficients=0    | Train R2=0.9183 | Test R2=0.9016
Alpha=0.1  | Zero coefficients=0    | Train R2=0.9182 | Test R2=0.9017
Alpha=1    | Zero coefficients=4    | Train R2=0.9166 | Test R2=0.8992
Alpha=10   | Zero coefficients=5    | Train R2=0.8062 | Test R2=0.7860
Alpha=100  | Zero coefficients=10   | Train R2=0.0000 | Test R2=-0.0161


In [24]:
# comparison
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score

models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(alpha=1),
    "Lasso": Lasso(alpha=1)
}

for name, model in models.items():
    model.fit(x_train, y_train)

    y_train_pred = model.predict(x_train)
    y_pred = model.predict(x_test)

    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_pred)

    print(
        f"{name:<20} | "
        f"Train R2={train_r2:.4f} | "
        f"Test R2={test_r2:.4f}"
    )

Linear Regression    | Train R2=0.9183 | Test R2=0.9015
Ridge                | Train R2=0.9183 | Test R2=0.9016
Lasso                | Train R2=0.9166 | Test R2=0.8992


# Lasso Regression

## 1. Definition

**Lasso Regression (Least Absolute Shrinkage and Selection Operator)** is a regularized version of Linear Regression that uses **L1 regularization**.

It helps:

* Reduce overfitting
* Control large coefficients
* Perform automatic feature selection

---

## 2. Why Lasso Regression?

In ordinary Linear Regression, the model minimizes prediction error. When there are many features, some coefficients may become unnecessarily large, which can increase model complexity and overfitting.

Lasso adds a penalty to the model's objective function to discourage large coefficients.

---

## 3. Lasso Objective Function

$$
\boxed{Loss = MSE + \alpha \sum |w_i|}
$$

Where:

* `MSE` → Mean Squared Error
* \(w_i\) → model coefficients
* \(\alpha\) → regularization strength
* \(\sum |w_i|\) → **L1 penalty**

The L1 penalty is the defining characteristic of Lasso.

---

## 4. Effect of `alpha`

`alpha` controls the strength of regularization.

### Small `alpha`

* Weak regularization
* Coefficients are changed less
* Fewer coefficients become zero
* Model can remain more complex

### Large `alpha`

* Strong regularization
* Coefficients are pushed closer to zero
* More coefficients can become exactly zero
* Model becomes simpler
* If `alpha` is too large → **underfitting**

```text
alpha ↑
   ↓
L1 penalty ↑
   ↓
Coefficients shrink
   ↓
More coefficients → 0
   ↓
Simpler model
```

---

## 5. Feature Selection ⭐

The key advantage of Lasso is that it can make coefficients **exactly zero**.

Example:

```text
Feature A → 4.5
Feature B → 1.2
Feature C → 0
Feature D → 3.7
Feature E → 0
```

Features C and E are effectively removed from the model.

Therefore:

> **Lasso performs automatic feature selection by shrinking some coefficients to zero.**

---

## 6. Lasso vs Ridge

| Feature                         | Linear Regression | Ridge                | Lasso       |     |   |
| ------------------------------- | ----------------- | -------------------- | ----------- | --- | - |
| Regularization                  | ❌                 | ✅                    | ✅           |     |   |
| Penalty                         | None              | L2                   | L1          |     |   |
| Penalty formula                 | —                 | \(\alpha\sum w_i^2\) | (\alpha\sum | w_i | ) |
| Shrinks coefficients            | ❌                 | ✅                    | ✅           |     |   |
| Can make coefficients exactly 0 | ❌                 | Usually ❌            | ✅           |     |   |
| Feature selection               | ❌                 | ❌                    | ✅           |     |   |

### Easy way to remember

> **Ridge → Shrinks**
> **Lasso → Shrinks + Selects**

---

## 7. Scaling and Lasso

Feature scaling is generally important when using Lasso because the regularization penalty operates on coefficient magnitudes.

A common workflow is:

```text
Data
 ↓
Train/Test Split
 ↓
StandardScaler
 ↓
Lasso
```

For datasets containing different feature types, use a `ColumnTransformer` and `Pipeline`.

---

## 8. Python Implementation

```python
from sklearn.linear_model import Lasso

lasso = Lasso(alpha=0.1)

lasso.fit(x_train, y_train)

y_pred = lasso.predict(x_test)
```

### Evaluate

```python
from sklearn.metrics import mean_squared_error, r2_score

print("MSE:", mean_squared_error(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))
```

### Check coefficients

```python
print(lasso.coef_)
```

Count zero coefficients:

```python
zero_coefficients = np.sum(lasso.coef_ == 0)

print("Zero coefficients:", zero_coefficients)
```

---

## 9. Choosing `alpha`

`alpha` should not be chosen simply because it produces more zero coefficients.

A very large `alpha` can remove useful features and cause underfitting.

The proper approach is to use **Cross-Validation** to select a suitable `alpha`.

```text
Different alpha values
        ↓
Cross-Validation
        ↓
Compare validation performance
        ↓
Choose suitable alpha
```

---

## 10. Our Experiment

We tested:

```text
alpha=0.01 → 0 zero coefficients → Test R² = 0.9016
alpha=0.1  → 0 zero coefficients → Test R² = 0.9017
alpha=1    → 4 zero coefficients → Test R² = 0.8992
alpha=10   → 5 zero coefficients → Test R² = 0.7860
alpha=100  → 10 zero coefficients → Test R² = -0.0161
```

This demonstrated:

* Increasing `alpha` increases regularization.
* More coefficients can become zero.
* Too much regularization causes underfitting.
* More feature removal does **not** automatically mean better performance.

---

## 11. Linear vs Ridge vs Lasso

Our comparison:

```text
Linear Regression → Train R² = 0.9183 | Test R² = 0.9015
Ridge             → Train R² = 0.9183 | Test R² = 0.9016
Lasso             → Train R² = 0.9166 | Test R² = 0.8992
```

The models performed similarly on this dataset.

The main reason to choose Lasso isn't necessarily a higher R². **Its major advantage is feature selection.**

---

## 12. Key Takeaways

```text
Lasso Regression
      ↓
Linear Regression + L1 Regularization
      ↓
Penalty = α Σ|wi|
      ↓
Shrinks coefficients
      ↓
Can make coefficients exactly 0
      ↓
Automatic Feature Selection
```

**Remember:**

* Lasso = **L1**
* `alpha` = regularization strength
* Higher `alpha` → stronger penalty
* Lasso can make coefficients **exactly zero**
* Zero coefficient → feature effectively excluded
* Too much `alpha` → underfitting
* Cross-validation should be used to select `alpha`
* Scaling is generally important for Lasso.
